# 02 · Cómo genera pacientes cada método

Cinco generadores propios, de lo más simple a lo más protector, y cuatro del extra `[deep]`
(synthcity). Aquí se ajustan sobre la mitad de entrenamiento de GBSG2 y se compara a simple
vista lo que producen.

In [1]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from synapse import datasets, generators, inference as inf, utility as ut, privacy as pv, compliance, viz
from synapse import experiment as ex
viz.style(); pd.set_option("display.precision", 3); pd.set_option("display.width", 140)
TABLES = ROOT / "results" / "tables"
metrics = pd.read_csv(TABLES / "study_metrics.csv") if (TABLES / "study_metrics.csv").exists() else None

In [2]:
d = datasets.load("gbsg2")
train, holdout = datasets.split_train_holdout(d.data, seed=0)
specs = {"bootstrap": {}, "marginal": {}, "copula": {}, "cart": {}, "dpcopula": {"epsilon": 1.0}}
syn = {g: generators.make(g, seed=0, **p).fit(train, d.schema).sample(len(train)) for g, p in specs.items()}
print("Disponibles en este entorno:", generators.available())

Disponibles en este entorno: ['bootstrap', 'marginal', 'copula', 'cart', 'dpcopula']


## Las mismas cinco filas, según cada generador

In [3]:
pd.concat({g: s.head(5) for g, s in syn.items()}, names=["generador", "fila"])

age  tsize  pnodes  progrec  estrec    time horTh menostat tgrade event
generador fila                                                                          
bootstrap 0     33.0   19.0     2.0      0.0     0.0  2128.0   yes      Pre     II     0
          1     59.0   30.0    13.0      7.0    81.0   410.0   yes     Post     II     1
          2     51.0   20.0     1.0     77.0    89.0  1753.0    no      Pre     II     1
          3     54.0   30.0     3.0      1.0     0.0  1089.0    no     Post    III     0
          4     67.0   25.0     1.0      6.0    19.0   529.0    no     Post     II     1
marginal  0     58.0   30.0    11.0    253.0   271.0  1643.0    no     Post     II     0
          1     47.0   13.0     3.0     33.0     8.0  1467.0   yes     Post    III     0
          2     34.0   20.0     1.0     16.0   805.0  1863.0    no     Post     II     0
          3     32.0   27.0     1.0      7.0     0.0   935.0   yes     Post     II     1
          4     63.0   23.0     3.0     10.0     1.0   756.0    no     Post    III     0
copula    0     60.0   60.0     5.0      1.0    44.0   971.0    no     Post      I     0
          1     54.0   24.0     2.0     46.0    38.0  2437.0    no      Pre      I     0
          2     62.0   25.0     2.0     45.0   529.0  2057.0   yes     Post     II     0
          3     42.0   30.0    11.0     14.0     8.0   572.0    no     Post     II     0
          4     57.0   30.0    12.0    186.0    10.0   784.0   yes     Post    III     1
cart      0     33.0   35.0     2.0      5.0    10.0   721.0   yes      Pre     II     0
          1     59.0   48.0     4.0     18.0    11.0   426.0   yes     Post    III     1
          2     51.0   25.0     1.0    167.0    38.0   195.0   yes      Pre     II     0
          3     54.0   27.0     5.0    401.0    47.0   692.0    no     Post     II     0
          4     67.0   30.0     1.0      0.0     0.0  2030.0    no     Post    III     0
dpcopula  0     40.0  105.0     9.0     90.0     3.0  3551.0   yes     Post    III     0
          1     44.0  157.0    30.0    422.0  1266.0   886.0   yes     Post     II     0
          2     56.0   78.0    21.0     15.0    44.0  2836.0    no     Post    III     0
          3     86.0   25.0    28.0     61.0   982.0  3244.0   yes     Post      I     0
          4     86.0   43.0    57.0    135.0  1504.0   282.0   yes     Post      I     0

## ¿Se conservan las distribuciones marginales?

In [4]:
uni = pd.DataFrame({g: ut.univariate(train, s, d.schema).set_index("column")["value"] for g, s in syn.items()})
uni.style.background_gradient(cmap="Blues", axis=None).format("{:.3f}")

,bootstrap,marginal,copula,cart,dpcopula
column,,,,,
age,0.029,0.061,0.032,0.029,0.337
tsize,0.044,0.029,0.020,0.029,0.666
pnodes,0.047,0.023,0.029,0.047,0.468
progrec,0.055,0.052,0.055,0.026,0.622
estrec,0.032,0.064,0.047,0.029,0.672
time,0.049,0.099,0.041,0.052,0.587
horTh,0.038,0.017,0.044,0.020,0.160
menostat,0.012,0.009,0.047,0.006,0.419
tgrade,0.026,0.017,0.038,0.017,0.113


## ¿Y las relaciones entre variables?

In [5]:
fig, axes = plt.subplots(1, 5, figsize=(17, 3.4))
real_assoc = ut.associations(train, d.schema)
for ax, (g, s) in zip(axes, syn.items()):
    diff = (ut.associations(s, d.schema) - real_assoc).abs()
    ax.imshow(diff, vmin=0, vmax=.5, cmap="Blues"); ax.set_title(f"{g}\n|Δ| medio = {ut.bivariate(train, s, d.schema)[0]:.3f}")
    ax.set_xticks([]); ax.set_yticks(range(len(diff)), diff.index, fontsize=7)
plt.tight_layout()

**Lectura.** El muestreo de marginales clava cada variable por separado pero destruye todas las
relaciones; la cópula y CART las conservan; la cópula con privacidad diferencial (ε = 1) paga el
ruido en ambas. Que una variable «se parezca» no garantiza que el análisis clínico se mantenga:
eso se mide en el cuaderno 03.